# 03.5 V2 — Pages Review (Pre-Generation Check)

Final review of `pages_for_generation.jsonl` before spending money on Gemini generation.
Eyeball page quality, type distribution, screenshot thumbnails, and HTML content.

**Sign off on this before running 06_v2.**

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
import re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse
from io import BytesIO
from IPython.display import Image as IPImage, HTML

PAGES_PATH = Path('../data/processed/pages_for_generation.jsonl')

pages = []
if PAGES_PATH.exists():
    with open(PAGES_PATH) as f:
        for line in f:
            pages.append(json.loads(line))
    print(f'Loaded {len(pages):,} pages')
    size_mb = PAGES_PATH.stat().st_size / 1e6
    print(f'File size: {size_mb:.1f} MB')
else:
    print(f'Not found: {PAGES_PATH}  — run 03_v2_html_extraction.ipynb first')

In [ ]:
if not pages:
    print('No pages yet — check back when 03_v2 finishes')
else:
    type_counts   = Counter(p['schema_type'] for p in pages)
    source_counts = Counter(p.get('source', 'unknown') for p in pages)

    # Text length stats
    def text_length(html):
        text = re.sub(r'<[^>]+>', ' ', html)
        return len(re.sub(r'\s+', ' ', text).strip())

    text_lengths = [text_length(p.get('html', '')) for p in pages]

    # TLD
    def get_tld(url):
        try:
            host = urlparse(url).netloc.lower().lstrip('www.')
            parts = host.split('.')
            if len(parts) >= 3 and parts[-2] in ('co', 'com', 'org', 'net'):
                return '.'.join(parts[-2:])
            return parts[-1]
        except Exception:
            return 'unknown'

    tld_counts = Counter(get_tld(p['url']) for p in pages)

    # Screenshot availability
    has_screenshot = sum(1 for p in pages if p.get('screenshot_path') and Path(p['screenshot_path']).exists())

    print(f'Total pages:     {len(pages):,}')
    print(f'Has screenshot:  {has_screenshot:,} ({has_screenshot/len(pages)*100:.0f}%)')
    print(f'Avg text length: {np.mean(text_lengths):,.0f} chars')
    print(f'Median text:     {np.median(text_lengths):,.0f} chars')
    print()
    print('TYPE DISTRIBUTION')
    for t, n in type_counts.most_common():
        bar = '█' * int(n / max(type_counts.values()) * 25)
        print(f'  {t:25s} {n:5,}  {bar}')

In [ ]:
# Summary charts
if pages:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('pages_for_generation.jsonl — Pre-Generation Review', fontsize=13, fontweight='bold')

    # Type distribution vs target
    ax = axes[0]
    target_path = Path('../config/type_distribution.json')
    if target_path.exists():
        with open(target_path) as f:
            cfg = json.load(f)
        targets = {k: v['weight'] for k, v in cfg['types'].items()}
        total_w = sum(targets.values())
        common  = sorted(set(targets) & set(type_counts))
        x = range(len(common))
        ax.bar([i - 0.2 for i in x], [targets[t] / total_w * 100 for t in common],
               width=0.38, label='Target %', color='lightcoral', alpha=0.85)
        total_pages = len(pages)
        ax.bar([i + 0.2 for i in x], [type_counts.get(t, 0) / total_pages * 100 for t in common],
               width=0.38, label='Actual %', color='steelblue', alpha=0.85)
        ax.set_xticks(list(x))
        ax.set_xticklabels(common, rotation=40, ha='right', fontsize=8)
        ax.set_ylabel('%')
        ax.set_title('Type distribution vs target')
        ax.legend(fontsize=9)
    else:
        types, tcounts = zip(*type_counts.most_common(10))
        ax.barh(range(len(types)), tcounts, color='steelblue')
        ax.set_yticks(range(len(types)))
        ax.set_yticklabels(types)
        ax.invert_yaxis()
        ax.set_title('@type distribution')

    # Text length histogram
    ax = axes[1]
    ax.hist(text_lengths, bins=40, color='steelblue', edgecolor='white',
            range=(0, min(10000, max(text_lengths))))
    ax.axvline(np.median(text_lengths), color='red', linestyle='--',
               label=f'Median {np.median(text_lengths):,.0f}')
    ax.set_xlabel('Text length (chars)')
    ax.set_ylabel('Pages')
    ax.set_title('Page text length distribution')
    ax.legend(fontsize=9)

    # Source + TLD breakdown
    ax = axes[2]
    top_tlds = tld_counts.most_common(8)
    tld_labels, tld_vals = zip(*top_tlds)
    ax.pie(tld_vals, labels=[f'.{t}' for t in tld_labels], autopct='%1.0f%%', startangle=90)
    ax.set_title('TLD breakdown')

    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    display(IPImage(data=buf.getvalue()))

In [ ]:
# Screenshot grid — eyeball a random sample of 9 pages
pages_with_shots = [p for p in pages if p.get('screenshot_path') and Path(p['screenshot_path']).exists()]

if pages_with_shots:
    sample = random.sample(pages_with_shots, min(9, len(pages_with_shots)))

    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    fig.suptitle('Random screenshot sample', fontsize=12, fontweight='bold')

    for ax, page in zip(axes.flat, sample):
        try:
            img = mpimg.imread(page['screenshot_path'])
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Load error', ha='center', va='center', transform=ax.transAxes)
        domain = urlparse(page['url']).netloc.replace('www.', '')
        ax.set_title(f"{page['schema_type']}\n{domain}", fontsize=7)
        ax.axis('off')

    for ax in axes.flat[len(sample):]:
        ax.axis('off')

    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    display(IPImage(data=buf.getvalue()))
else:
    print('No screenshots available yet')

In [ ]:
# Sample HTML snippets per type — check content quality
if pages:
    by_type = {}
    for p in pages:
        by_type.setdefault(p['schema_type'], []).append(p)

    def extract_text(html, max_chars=400):
        html = re.sub(r'<(script|style)[^>]*>.*?</(script|style)>', '', html, flags=re.DOTALL | re.IGNORECASE)
        text = re.sub(r'<[^>]+>', ' ', html)
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars] + '...' if len(text) > max_chars else text

    print('CONTENT SAMPLE (1 page per type)')
    print('=' * 70)
    for t in sorted(by_type):
        page = random.choice(by_type[t])
        domain = urlparse(page['url']).netloc.replace('www.', '')
        snippet = extract_text(page.get('html', ''))
        print(f'\n{t} — {domain}')
        print(f'  URL: {page["url"]}')
        print(f'  Text: {snippet}')

In [ ]:
# Cost estimate for generation
if pages:
    # Rough token estimate: ~2K tokens per page (HTML) + ~500 screenshot tokens
    input_tokens_per_page  = 2_500
    output_tokens_per_page = 800   # JSON-LD output

    # Gemini 2.5 Flash pricing (as of 2026-03)
    input_cost_per_1m  = 0.15   # $0.15 per 1M input tokens
    output_cost_per_1m = 0.60   # $0.60 per 1M output tokens

    total_input  = len(pages) * input_tokens_per_page
    total_output = len(pages) * output_tokens_per_page

    cost_standard = (total_input / 1e6 * input_cost_per_1m +
                     total_output / 1e6 * output_cost_per_1m)
    cost_batch    = cost_standard * 0.5  # 50% discount

    print('GENERATION COST ESTIMATE')
    print('=' * 40)
    print(f'Pages:           {len(pages):,}')
    print(f'Est input tokens: {total_input/1e6:.1f}M')
    print(f'Est output tokens:{total_output/1e6:.1f}M')
    print()
    print(f'Standard API:    ${cost_standard:.2f}')
    print(f'Batch API (-50%): ${cost_batch:.2f}')
    print()
    print('→ When happy with the data above, run 06_v2_synthetic_generation.ipynb')